### Question 1. A

In [1]:
import numpy as np
from numpy.linalg import norm, solve, multi_dot
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

In [7]:
f = lambda x,y: 100*(y-x**2)**2+(1-x)**2
Df = lambda x,y: np.array([400*x**3-400*x*y-2+2*x,
                           200*y - 200*x**2])

In [8]:
def PR_beta(Dfk,Dfk1, pk):
    return ((Dfk1-Dfk)@Dfk1) / (Dfk@Dfk)

In [9]:
def FR_beta(Dfk,Dfk1, pk): 
    return (Dfk1@Dfk1) / (Dfk@Dfk)

In [12]:
%%time
def nonlinearCG(x0, y0, f, Df, beta, restart=True, tol=1e-8, max_steps=10000):
    x,y = x0,y0
    path = [[x,y]]
    i=0                   
    dx = Df(x,y)          
    pk = -dx
    while norm(dx) > tol and i<max_steps:
        def subproblem1D(alpha): #exact line search
            return f(x + alpha * pk[0],y+alpha*pk[1])
        res = minimize_scalar(subproblem1D)
        alpha = res.x      
        xnew,ynew = x + alpha*pk[0], y + alpha*pk[1]
        dx1 = Df(xnew,ynew)      # Df_{k+1}
        bk = beta(dx,dx1,pk)  # beta_k
        if restart and i%2 == 0:
            bk=0
        pk = -dx1 + bk*pk
        path.append([xnew,ynew])
        x,y = xnew,ynew
        dx = dx1
        i += 1
    return np.array(path), i, x, y

CPU times: user 11 µs, sys: 1 µs, total: 12 µs
Wall time: 16 µs


In [13]:
x0 = 1.2; y0=1.2
path_PR, i, x, y = nonlinearCG(x0, y0, f, Df, PR_beta, False)
print(f'After {i} iterations, approximate minimum is {f(x,y)} at {x,y}')

After 10 iterations, approximate minimum is 2.154344126455914e-25 at (1.0000000000004168, 1.000000000000854)
